# TFM

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import seasonal_decompose
import warnings
warnings.filterwarnings("ignore")

In [2]:
df_daily = pd.read_csv("data_dsmarket/daily_calendar_with_events.csv")
df_prices = pd.read_csv('data_dsmarket/item_prices.csv')
df_sales = pd.read_csv('data_dsmarket/item_sales.csv')

## DAILY

In [3]:
df_daily.columns

Index(['date', 'weekday', 'weekday_int', 'd', 'event'], dtype='object')

In [4]:
df_daily.isna().sum()

date              0
weekday           0
weekday_int       0
d                 0
event          1887
dtype: int64

In [5]:
display(df_daily.head())
df_daily.describe(include= 'all')

,date,weekday,weekday_int,d,event
0,2011-01-29,Saturday,1,d_1,NaN
1,2011-01-30,Sunday,2,d_2,NaN
2,2011-01-31,Monday,3,d_3,NaN
3,2011-02-01,Tuesday,4,d_4,NaN
4,2011-02-02,Wednesday,5,d_5,NaN


,date,weekday,weekday_int,d,event
count,1913,1913,1913.000000,1913,26
unique,1913,7,NaN,1913,5
top,2011-01-29,Saturday,NaN,d_1,SuperBowl
freq,1,274,NaN,1,6
mean,NaN,NaN,3.997386,NaN,NaN
std,NaN,NaN,2.001175,NaN,NaN
min,NaN,NaN,1.000000,NaN,NaN
25%,NaN,NaN,2.000000,NaN,NaN
50%,NaN,NaN,4.000000,NaN,NaN
75%,NaN,NaN,6.000000,NaN,NaN


In [6]:
df_daily.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1913 entries, 0 to 1912
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   date         1913 non-null   object
 1   weekday      1913 non-null   object
 2   weekday_int  1913 non-null   int64 
 3   d            1913 non-null   object
 4   event        26 non-null     object
dtypes: int64(1), object(4)
memory usage: 74.9+ KB


In [7]:
df_daily['date'] = pd.to_datetime( df_daily['date']) 
df_daily = df_daily.sort_values(by = 'date') # como esta ordenado luego me tiene que funcionar directamente lo de comprobar si esta completo 

In [8]:
print(df_daily['date'].min())
print(df_daily['date'].max())
print(df_daily['date'].max() - df_daily['date'].min())
# creo que esta completo porque son 1913 registros pero vamos a comprobarlo

2011-01-29 00:00:00
2016-04-24 00:00:00
1912 days 00:00:00


In [9]:
def comprobar_fechas(c):
    inicio = c.min()
    fin = c.max()
    completo = pd.date_range(inicio, fin)
    print((completo == c).all())

comprobar_fechas(df_daily['date'])

True


In [10]:
df_daily['event'].value_counts(dropna= False)
# vemos que la mayoría de días no hay ningún evento. A lo mejor podríamos incorporar más información al calendario, como por ejemplo si llovío o el tiempo, pero para más adelante cuadno tengamos un df bien construido 

NaN               1887
SuperBowl            6
Ramadan starts       5
Thanksgiving         5
NewYear              5
Easter               5
Name: event, dtype: int64

In [11]:
display(df_daily[df_daily['event'] == 'Ramadan starts'])
display(df_daily[df_daily['event'] == 'SuperBowl'])
display(df_daily[df_daily['event'] == 'Thanksgiving'])
display(df_daily[df_daily['event'] == 'NewYear'])
display(df_daily[df_daily['event'] == 'Easter'])
# aquí lo que podemos apreciar es que tenemos una fecha por año y que no siempre coincide la fecha en la cae el evento todos los años, excepto en el caso de Año Nuevo. 
# Tenemos 5 registros de cada uno menos de la superbowl porque al ser en febrero lo tenemos tanto en la parte de 2011 (a partir de febrero prácticamente) como en la parte de 2016 (hasta marzo)

,date,weekday,weekday_int,d,event
184,2011-08-01,Monday,3,d_185,Ramadan starts
538,2012-07-20,Friday,7,d_539,Ramadan starts
892,2013-07-09,Tuesday,4,d_893,Ramadan starts
1247,2014-06-29,Sunday,2,d_1248,Ramadan starts
1601,2015-06-18,Thursday,6,d_1602,Ramadan starts


,date,weekday,weekday_int,d,event
8,2011-02-06,Sunday,2,d_9,SuperBowl
372,2012-02-05,Sunday,2,d_373,SuperBowl
736,2013-02-03,Sunday,2,d_737,SuperBowl
1100,2014-02-02,Sunday,2,d_1101,SuperBowl
1464,2015-02-01,Sunday,2,d_1465,SuperBowl
1835,2016-02-07,Sunday,2,d_1836,SuperBowl


,date,weekday,weekday_int,d,event
299,2011-11-24,Thursday,6,d_300,Thanksgiving
663,2012-11-22,Thursday,6,d_664,Thanksgiving
1034,2013-11-28,Thursday,6,d_1035,Thanksgiving
1398,2014-11-27,Thursday,6,d_1399,Thanksgiving
1762,2015-11-26,Thursday,6,d_1763,Thanksgiving


,date,weekday,weekday_int,d,event
337,2012-01-01,Sunday,2,d_338,NewYear
703,2013-01-01,Tuesday,4,d_704,NewYear
1068,2014-01-01,Wednesday,5,d_1069,NewYear
1433,2015-01-01,Thursday,6,d_1434,NewYear
1798,2016-01-01,Friday,7,d_1799,NewYear


,date,weekday,weekday_int,d,event
435,2012-04-08,Sunday,2,d_436,Easter
792,2013-03-31,Sunday,2,d_793,Easter
1177,2014-04-20,Sunday,2,d_1178,Easter
1527,2015-04-05,Sunday,2,d_1528,Easter
1884,2016-03-27,Sunday,2,d_1885,Easter


In [12]:
(df_daily['d'].str[2:].astype('int') == pd.Series(range(1,1914))).all()
# la columna d son los numeros normales ordenados y no falta ninguno. 

True

In [13]:
df_daily.groupby('weekday')['weekday_int'].value_counts() # una de las dos la quitaría 

weekday    weekday_int
Friday     7              273
Monday     3              273
Saturday   1              274
Sunday     2              274
Thursday   6              273
Tuesday    4              273
Wednesday  5              273
Name: weekday_int, dtype: int64

In [17]:
df_daily['yearweek'] = df_daily['date'].dt.isocalendar().year.astype(str) +  df_daily['date'].dt.isocalendar().week.astype(str).str.zfill(2)
df_daily['yearweek'] = df_daily['yearweek'].astype('float')

In [18]:
df_daily

,date,weekday,weekday_int,d,event,yearweek
0,2011-01-29,Saturday,1,d_1,NaN,201104.0
1,2011-01-30,Sunday,2,d_2,NaN,201104.0
2,2011-01-31,Monday,3,d_3,NaN,201105.0
3,2011-02-01,Tuesday,4,d_4,NaN,201105.0
4,2011-02-02,Wednesday,5,d_5,NaN,201105.0
...,...,...,...,...,...,...
1908,2016-04-20,Wednesday,5,d_1909,NaN,201616.0
1909,2016-04-21,Thursday,6,d_1910,NaN,201616.0
1910,2016-04-22,Friday,7,d_1911,NaN,201616.0
1911,2016-04-23,Saturday,1,d_1912,NaN,201616.0


## SALES

In [19]:
display(df_sales.head())
df_sales.describe(include= 'all')

,id,item,category,department,store,store_code,region,d_1,d_2,d_3,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,ACCESORIES_1_001_NYC_1,ACCESORIES_1_001,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,0,0,0,...,1,3,0,1,1,1,3,0,1,1
1,ACCESORIES_1_002_NYC_1,ACCESORIES_1_002,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,ACCESORIES_1_003_NYC_1,ACCESORIES_1_003,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,0,0,0,...,2,1,2,1,1,1,0,1,1,1
3,ACCESORIES_1_004_NYC_1,ACCESORIES_1_004,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,0,0,0,...,1,0,5,4,1,0,1,3,7,2
4,ACCESORIES_1_005_NYC_1,ACCESORIES_1_005,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,0,0,0,...,2,1,1,0,1,1,2,2,2,4


,id,item,category,department,store,store_code,region,d_1,d_2,d_3,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
count,30490,30490,30490,30490,30490,30490,30490,30490.000000,30490.000000,30490.000000,...,30490.000000,30490.000000,30490.000000,30490.000000,30490.000000,30490.000000,30490.000000,30490.000000,30490.000000,30490.000000
unique,30490,3049,3,7,10,10,3,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,ACCESORIES_1_001_NYC_1,ACCESORIES_1_001,SUPERMARKET,SUPERMARKET_3,Greenwich_Village,NYC_1,New York,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,10,14370,8230,3049,3049,12196,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.070220,1.041292,0.780026,...,1.370581,1.586159,1.693670,1.248245,1.232207,1.159167,1.149000,1.328862,1.605838,1.633158
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.126689,5.365468,3.667454,...,3.740017,4.097191,4.359809,3.276925,3.125471,2.876026,2.950364,3.358012,4.089422,3.812248
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,...,1.000000,2.000000,2.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2.000000,2.000000


In [20]:
df_sales_melt = pd.melt(df_sales, id_vars= df_sales.columns[:7], var_name= 'd', value_name = 'sales')
df_sales_melt

,id,item,category,department,store,store_code,region,d,sales
0,ACCESORIES_1_001_NYC_1,ACCESORIES_1_001,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0
1,ACCESORIES_1_002_NYC_1,ACCESORIES_1_002,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0
2,ACCESORIES_1_003_NYC_1,ACCESORIES_1_003,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0
3,ACCESORIES_1_004_NYC_1,ACCESORIES_1_004,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0
4,ACCESORIES_1_005_NYC_1,ACCESORIES_1_005,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0
...,...,...,...,...,...,...,...,...,...
58327365,SUPERMARKET_3_823_PHI_3,SUPERMARKET_3_823,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1913,1
58327366,SUPERMARKET_3_824_PHI_3,SUPERMARKET_3_824,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1913,0
58327367,SUPERMARKET_3_825_PHI_3,SUPERMARKET_3_825,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1913,0
58327368,SUPERMARKET_3_826_PHI_3,SUPERMARKET_3_826,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1913,3


In [21]:
# vamos a ver si hay nulos
df_sales.isna().sum().sum()

0

In [23]:
df_sales.store_code.value_counts()
df_sales.store.value_counts()
df_sales.groupby('store')['item'].count()
df_sales.category.value_counts()
df_sales.region.value_counts()
# El dataset tiene 3049 productos, de 3 categorías y 7 departamentos, vendidos en 10 tiendas de 3 estados

NYC_1    3049
NYC_2    3049
NYC_3    3049
NYC_4    3049
BOS_1    3049
BOS_2    3049
BOS_3    3049
PHI_1    3049
PHI_2    3049
PHI_3    3049
Name: store_code, dtype: int64

New York        12196
Boston           9147
Philadelphia     9147
Name: region, dtype: int64

In [ ]:
df_sales_melt

## PRECIOS

In [24]:
df_prices = df_prices.sort_values(by = 'yearweek')

In [25]:
df_prices.head()

,item,category,store_code,yearweek,sell_price
5539160,SUPERMARKET_3_702,SUPERMARKET,PHI_1,201105.0,3.9360
6613947,HOME_&_GARDEN_2_445,HOME_&_GARDEN,PHI_3,201105.0,8.1000
261069,HOME_&_GARDEN_2_041,HOME_&_GARDEN,NYC_1,201105.0,6.5875
1516539,HOME_&_GARDEN_1_117,HOME_&_GARDEN,NYC_3,201105.0,11.2125
563249,SUPERMARKET_3_188,SUPERMARKET,NYC_1,201105.0,2.3760


In [26]:
df_prices.item.value_counts()
# tenemso los 3049 productos. como tenemos muchso recuentos distintos y para cada producto el precio en 2078 semanas

SUPERMARKET_3_702      2870
HOME_&_GARDEN_1_234    2870
ACCESORIES_1_194       2870
SUPERMARKET_1_168      2870
SUPERMARKET_3_590      2870
                       ... 
HOME_&_GARDEN_1_308     652
HOME_&_GARDEN_1_159     633
HOME_&_GARDEN_1_242     610
SUPERMARKET_3_296       602
SUPERMARKET_2_379       543
Name: item, Length: 3049, dtype: int64

In [27]:
df_prices.isna().sum()

item               0
category           0
store_code         0
yearweek      243920
sell_price         0
dtype: int64

In [28]:
df_prices

,item,category,store_code,yearweek,sell_price
5539160,SUPERMARKET_3_702,SUPERMARKET,PHI_1,201105.0,3.9360
6613947,HOME_&_GARDEN_2_445,HOME_&_GARDEN,PHI_3,201105.0,8.1000
261069,HOME_&_GARDEN_2_041,HOME_&_GARDEN,NYC_1,201105.0,6.5875
1516539,HOME_&_GARDEN_1_117,HOME_&_GARDEN,NYC_3,201105.0,11.2125
563249,SUPERMARKET_3_188,SUPERMARKET,NYC_1,201105.0,2.3760
...,...,...,...,...,...
6965701,SUPERMARKET_3_827,SUPERMARKET,PHI_3,NaN,1.2000
6965702,SUPERMARKET_3_827,SUPERMARKET,PHI_3,NaN,1.2000
6965703,SUPERMARKET_3_827,SUPERMARKET,PHI_3,NaN,1.2000
6965704,SUPERMARKET_3_827,SUPERMARKET,PHI_3,NaN,1.2000


In [29]:
len(df_prices[ (df_prices['item'] == 'SUPERMARKET_3_702') & (df_prices['yearweek'] == 201105) ]) == len(df_prices[ (df_prices['item'] == 'SUPERMARKET_3_702') & (df_prices['yearweek'] == 201625) ])
# vemos que el df tiene distinto número de tiendas en la semana 201105 que en la semana 201625 para el mismo articulo, lo que ya demuestra que no siempre está disponible en las mismas tiendas en todas las semanas.

False

## CRUZAMOS LOS DATASETS

In [30]:
df = df_sales_melt.merge(df_daily, on='d', how='left')
df = df.merge(df_prices.drop(columns='category'), on=['item', 'store_code', 'yearweek'], how='left')

In [31]:
def estacion(mes):
    if mes in [12, 1, 2]:
        return 'Invierno'
    elif mes in [3, 4, 5]:
        return 'Primavera'
    elif mes in [6, 7, 8]:
        return 'Verano'
    else:
        return 'Otoño'

df['season'] = df['date'].dt.month.apply(estacion)

In [32]:
df

,id,item,category,department,store,store_code,region,d,sales,date,weekday,weekday_int,event,yearweek,sell_price,season
0,ACCESORIES_1_001_NYC_1,ACCESORIES_1_001,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0,2011-01-29,Saturday,1,NaN,201104.0,NaN,Invierno
1,ACCESORIES_1_002_NYC_1,ACCESORIES_1_002,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0,2011-01-29,Saturday,1,NaN,201104.0,NaN,Invierno
2,ACCESORIES_1_003_NYC_1,ACCESORIES_1_003,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0,2011-01-29,Saturday,1,NaN,201104.0,NaN,Invierno
3,ACCESORIES_1_004_NYC_1,ACCESORIES_1_004,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0,2011-01-29,Saturday,1,NaN,201104.0,NaN,Invierno
4,ACCESORIES_1_005_NYC_1,ACCESORIES_1_005,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0,2011-01-29,Saturday,1,NaN,201104.0,NaN,Invierno
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58327365,SUPERMARKET_3_823_PHI_3,SUPERMARKET_3_823,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1913,1,2016-04-24,Sunday,2,NaN,201616.0,3.576,Primavera
58327366,SUPERMARKET_3_824_PHI_3,SUPERMARKET_3_824,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1913,0,2016-04-24,Sunday,2,NaN,201616.0,2.976,Primavera
58327367,SUPERMARKET_3_825_PHI_3,SUPERMARKET_3_825,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1913,0,2016-04-24,Sunday,2,NaN,201616.0,4.776,Primavera
58327368,SUPERMARKET_3_826_PHI_3,SUPERMARKET_3_826,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1913,3,2016-04-24,Sunday,2,NaN,201616.0,1.536,Primavera


## EXPLORACIÓN

In [34]:
muestra_producto_tienda = df [ df.id == 'SUPERMARKET_3_594_NYC_2'] # son las ventas de un producto en una tienda solamente
px.line(muestra_producto_tienda, x = 'date', y = 'sales')

In [35]:
muestra_producto = df[df.item == 'SUPERMARKET_3_594'].groupby('date').sales.sum().reset_index()
muestra_producto['date'] = pd.to_datetime(muestra_producto['date'])
fig = px.line(muestra_producto, x='date', y='sales')
eventos = df[df['event'].notna()]
eventos['date'] = pd.to_datetime(eventos['date'])
for fecha in eventos['date'].unique():
    fig.add_vline(x=pd.Timestamp(fecha).timestamp() * 1000, line_dash="dash", line_color="red", opacity=0.5)
fig.show()

In [37]:
# Vamos a ver cuanto vende cada tienda
df.groupby(by = ['store_code','date'])['sales'].sum().reset_index()

,store_code,date,sales
0,BOS_1,2011-01-29,2556
1,BOS_1,2011-01-30,2687
2,BOS_1,2011-01-31,1822
3,BOS_1,2011-02-01,2258
4,BOS_1,2011-02-02,1694
...,...,...,...
19125,PHI_3,2016-04-20,3159
19126,PHI_3,2016-04-21,3226
19127,PHI_3,2016-04-22,3828
19128,PHI_3,2016-04-23,4686


In [38]:
fig = px.line(df.groupby(by = ['store_code','date'])['sales'].sum().reset_index(), x = 'date', y = 'sales', color = 'store_code')
for fecha in eventos['date'].unique():
    fig.add_vline(x=pd.Timestamp(fecha).timestamp() * 1000, line_dash="dash", line_color="red", opacity=0.5)
fig.show()

In [42]:
df[(df.date.dt.month == 12) & (df.date.dt.day == 25) & (df.sales != 0)]

,id,item,category,department,store,store_code,region,d,sales,date,weekday,weekday_int,event,yearweek,sell_price,season
10067559,SUPERMARKET_3_586_NYC_2,SUPERMARKET_3_586,SUPERMARKET,SUPERMARKET_3,Harlem,NYC_2,New York,d_331,6,2011-12-25,Sunday,2,NaN,201151.0,1.7760,Invierno
10067725,SUPERMARKET_3_755_NYC_2,SUPERMARKET_3_755,SUPERMARKET,SUPERMARKET_3,Harlem,NYC_2,New York,d_331,1,2011-12-25,Sunday,2,NaN,201151.0,1.4160,Invierno
10070428,SUPERMARKET_3_406_NYC_3,SUPERMARKET_3_406,SUPERMARKET,SUPERMARKET_3,Tribeca,NYC_3,New York,d_331,1,2011-12-25,Sunday,2,NaN,201151.0,0.9840,Invierno
10082362,SUPERMARKET_3_144_BOS_3,SUPERMARKET_3_144,SUPERMARKET,SUPERMARKET_3,Back_Bay,BOS_3,Boston,d_331,1,2011-12-25,Sunday,2,NaN,201151.0,2.2560,Invierno
10082444,SUPERMARKET_3_226_BOS_3,SUPERMARKET_3_226,SUPERMARKET,SUPERMARKET_3,Back_Bay,BOS_3,Boston,d_331,1,2011-12-25,Sunday,2,NaN,201151.0,1.7760,Invierno
10088542,SUPERMARKET_3_226_PHI_2,SUPERMARKET_3_226,SUPERMARKET,SUPERMARKET_3,Yorktown,PHI_2,Philadelphia,d_331,1,2011-12-25,Sunday,2,NaN,201151.0,1.7760,Invierno
10088554,SUPERMARKET_3_238_PHI_2,SUPERMARKET_3_238,SUPERMARKET,SUPERMARKET_3,Yorktown,PHI_2,Philadelphia,d_331,1,2011-12-25,Sunday,2,NaN,201151.0,4.1760,Invierno
10088577,SUPERMARKET_3_261_PHI_2,SUPERMARKET_3_261,SUPERMARKET,SUPERMARKET_3,Yorktown,PHI_2,Philadelphia,d_331,1,2011-12-25,Sunday,2,NaN,201151.0,5.3760,Invierno
21226868,SUPERMARKET_3_555_NYC_2,SUPERMARKET_3_555,SUPERMARKET,SUPERMARKET_3,Harlem,NYC_2,New York,d_697,1,2012-12-25,Tuesday,4,NaN,201252.0,1.8960,Invierno
21226899,SUPERMARKET_3_586_NYC_2,SUPERMARKET_3_586,SUPERMARKET,SUPERMARKET_3,Harlem,NYC_2,New York,d_697,1,2012-12-25,Tuesday,4,NaN,201252.0,1.8960,Invierno


In [40]:
fig = px.bar(df.groupby(['category', 'store_code'])['sales'].sum().reset_index(), x='category', y='sales', color='store_code', barmode='stack')
fig.show()

In [41]:
ventas_dia = df.groupby(by = ['store_code', 'weekday'])['sales'].sum().reset_index()
ventas_dia = ventas_dia.pivot(index = 'weekday', columns = 'store_code', values = 'sales')
fig = px.imshow(ventas_dia, color_continuous_scale = 'viridis')
fig.show()

In [43]:
fig = px.box(df.groupby(['store_code', 'date'])['sales'].sum().reset_index(), x='store_code', y='sales', color='store_code')
fig.show()

## PRODUCTOS

### Top productos por ciudad

In [45]:
top_productos = df.groupby('item')['sales'].sum().sort_values(ascending=False).head(10)
print("top productos en general:")
print(top_productos)

for region in ['New York', 'Boston', 'Philadelphia']:
    print(f"\nTop 10 en {region}:")
    top_region = df[df['region'] == region].groupby('item')['sales'].sum().sort_values(ascending=False).head(10)
    print(top_region)

top productos en general:
item
SUPERMARKET_3_090    1002529
SUPERMARKET_3_586     920242
SUPERMARKET_3_252     565299
SUPERMARKET_3_555     491287
SUPERMARKET_3_714     396172
SUPERMARKET_3_587     396119
SUPERMARKET_3_694     390001
SUPERMARKET_3_226     363082
SUPERMARKET_3_202     295689
SUPERMARKET_3_723     284333
Name: sales, dtype: int64

Top 10 en New York:
item
SUPERMARKET_3_090    486138
SUPERMARKET_3_586    318050
SUPERMARKET_3_252    237172
SUPERMARKET_3_120    176446
SUPERMARKET_3_587    166065
SUPERMARKET_3_808    165778
SUPERMARKET_3_635    158166
SUPERMARKET_3_541    156589
SUPERMARKET_3_714    146000
SUPERMARKET_3_555    141146
Name: sales, dtype: int64

Top 10 en Boston:
item
SUPERMARKET_3_586    455411
SUPERMARKET_3_090    328034
SUPERMARKET_3_252    257079
SUPERMARKET_3_555    244685
SUPERMARKET_3_377    164976
SUPERMARKET_3_587    138710
SUPERMARKET_3_714    129772
SUPERMARKET_3_202    123263
SUPERMARKET_3_030    109496
SUPERMARKET_3_694    104541
Name: sales, dtyp

### Productos en declive

In [54]:
# Para eso vamos a agrupar por meses para ver cuales van disminuyendo
df['year_month'] = df['date'].dt.to_period('M')
ventas_mensuales = df.groupby(by = ['item', 'year_month']).sales.sum().reset_index()
ventas_mensuales['year_month'] = ventas_mensuales['year_month'].astype(str)
df

,id,item,category,department,store,store_code,region,d,sales,date,weekday,weekday_int,event,yearweek,sell_price,season,year_month
0,ACCESORIES_1_001_NYC_1,ACCESORIES_1_001,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0,2011-01-29,Saturday,1,NaN,201104.0,NaN,Invierno,2011-01
1,ACCESORIES_1_002_NYC_1,ACCESORIES_1_002,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0,2011-01-29,Saturday,1,NaN,201104.0,NaN,Invierno,2011-01
2,ACCESORIES_1_003_NYC_1,ACCESORIES_1_003,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0,2011-01-29,Saturday,1,NaN,201104.0,NaN,Invierno,2011-01
3,ACCESORIES_1_004_NYC_1,ACCESORIES_1_004,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0,2011-01-29,Saturday,1,NaN,201104.0,NaN,Invierno,2011-01
4,ACCESORIES_1_005_NYC_1,ACCESORIES_1_005,ACCESORIES,ACCESORIES_1,Greenwich_Village,NYC_1,New York,d_1,0,2011-01-29,Saturday,1,NaN,201104.0,NaN,Invierno,2011-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58327365,SUPERMARKET_3_823_PHI_3,SUPERMARKET_3_823,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1913,1,2016-04-24,Sunday,2,NaN,201616.0,3.576,Primavera,2016-04
58327366,SUPERMARKET_3_824_PHI_3,SUPERMARKET_3_824,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1913,0,2016-04-24,Sunday,2,NaN,201616.0,2.976,Primavera,2016-04
58327367,SUPERMARKET_3_825_PHI_3,SUPERMARKET_3_825,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1913,0,2016-04-24,Sunday,2,NaN,201616.0,4.776,Primavera,2016-04
58327368,SUPERMARKET_3_826_PHI_3,SUPERMARKET_3_826,SUPERMARKET,SUPERMARKET_3,Queen_Village,PHI_3,Philadelphia,d_1913,3,2016-04-24,Sunday,2,NaN,201616.0,1.536,Primavera,2016-04


In [47]:
producto_ejemplo = ventas_mensuales['item'].iloc[120] # este por ejemplo tiene siempre picos en diciembre
data_ejemplo = ventas_mensuales[ventas_mensuales['item'] == producto_ejemplo]
fig = px.line(data_ejemplo, x='year_month', y='sales', title=f'Ventas mensuales: {producto_ejemplo}')
fig.show()

In [55]:
def calcular_tendencia_producto(df):
    try:
        if len(df) < 24: 
            return np.nan
        resultado = seasonal_decompose(df['sales'], model = 'multiplicative', period=6)
        tendencia = resultado.trend.dropna() #quito el principio y el final pq se me quedan muchos nulos
        x = np.arange(len(tendencia))
        pendiente = np.polyfit(x, tendencia, 1)[0]
        return pendiente
    except:
        return np.nan
    
    
tendencias_productos = []
for item in ventas_mensuales['item'].unique():
    producto = ventas_mensuales[ventas_mensuales['item'] == item].sort_values('year_month')
    pendiente = calcular_tendencia_producto(producto)
    tendencias_productos.append({
        'item': item,
        'pendiente_tendencia': pendiente,
        'ventas_totales': producto['sales'].sum()
    })
df_tendencias = pd.DataFrame(tendencias_productos)
df_tendencias = df_tendencias.dropna()
df_tendencias = df_tendencias.sort_values('pendiente_tendencia')
print("productos con pendiente + negativa:")
print(df_tendencias.head(20))

productos con pendiente + negativa:
                   item  pendiente_tendencia  ventas_totales
2374  SUPERMARKET_3_150           -96.357509          147532
2186  SUPERMARKET_2_360           -78.836104          257119
2779  SUPERMARKET_3_555           -70.333456          491287
2810  SUPERMARKET_3_586           -63.211339          920242
2859  SUPERMARKET_3_635           -56.388234          282134
1954  SUPERMARKET_2_128           -55.132040          149465
1848  SUPERMARKET_2_021           -43.940724          121679
2378  SUPERMARKET_3_154           -43.141725          101583
2670  SUPERMARKET_3_446           -41.866947           96285
2692  SUPERMARKET_3_468           -39.319181           85612
2304  SUPERMARKET_3_080           -37.797090          262650
2600  SUPERMARKET_3_376           -36.198017          228551
2102  SUPERMARKET_2_276           -34.874125          158887
3029  SUPERMARKET_3_808           -32.777090          281879
2543  SUPERMARKET_3_319           -29.362151     

In [57]:
productos_declive = df_tendencias.head(10)
fig = make_subplots(rows=5, cols=2, subplot_titles=[f"{prod}" for prod, pend in zip(productos_declive['item'], productos_declive['pendiente_tendencia'])])

for idx, prod in enumerate(productos_declive['item']):
    datos_prod = ventas_mensuales[ventas_mensuales['item'] == prod].sort_values('year_month')
    fig.add_trace(go.Scatter(x=datos_prod['year_month'], y=datos_prod['sales'], mode='lines+markers', name=prod, showlegend=False), row=idx//2+1, col=idx%2+1)

fig.update_layout(height=1200, title_text="Top 10 Productos en Declive")
fig.show()

### productos con mas diferencia de precios entre tiendas

In [50]:
precios = df.groupby('item')['sell_price'].describe() # tenemos los 3049 productos, vamos a quedarnos con los q presentasen mayor varianza en los precios
precios['rango'] = precios['max'] - precios['min']
precios.sort_values('rango', ascending=False).head(20)

,count,mean,std,min,25%,50%,75%,max,rango
item,,,,,,,,,
HOME_&_GARDEN_2_406,19019.0,16.057278,5.971576,4.0750,15.5750,15.5750,15.5875,134.1500,130.0750
HOME_&_GARDEN_2_466,18382.0,9.408073,5.439682,7.4625,8.7125,8.7125,8.7125,65.7750,58.3125
HOME_&_GARDEN_2_178,18361.0,8.744167,1.590913,3.7500,8.7125,8.7125,8.7125,55.4500,51.7000
HOME_&_GARDEN_2_250,19012.0,8.797865,1.379232,4.2000,8.7000,8.7000,8.7125,42.7250,38.5250
HOME_&_GARDEN_2_514,19019.0,23.464805,1.153527,1.2500,22.4250,23.7125,23.7125,26.2125,24.9625
ACCESORIES_1_342,13818.0,26.290920,1.230989,2.6600,25.2301,27.1054,27.1054,27.1054,24.4454
HOME_&_GARDEN_1_469,18844.0,22.467738,0.883188,1.2500,22.4625,22.4625,22.4625,25.5750,24.3250
ACCESORIES_1_257,18340.0,22.346447,0.518051,10.5868,22.2908,22.2908,22.2908,34.3140,23.7272
HOME_&_GARDEN_1_342,7917.0,22.391921,0.916852,1.2500,22.4000,22.4625,22.4625,22.4625,21.2125


In [51]:
precios.sort_values(by = 'std', ascending = False).head(10)

,count,mean,std,min,25%,50%,75%,max,rango
item,,,,,,,,,
HOME_&_GARDEN_2_406,19019.0,16.057278,5.971576,4.0750,15.5750,15.5750,15.5875,134.1500,130.0750
HOME_&_GARDEN_2_466,18382.0,9.408073,5.439682,7.4625,8.7125,8.7125,8.7125,65.7750,58.3125
HOME_&_GARDEN_2_292,18956.0,15.803527,3.149647,12.4625,12.4625,18.7125,18.7125,19.9625,7.5000
ACCESORIES_1_060,13783.0,39.486032,2.069043,34.5800,39.8468,39.8601,41.2034,41.2034,6.6234
ACCESORIES_1_361,13853.0,39.485912,2.065196,35.8834,39.8468,39.8601,41.2034,41.2034,5.3200
ACCESORIES_1_225,13853.0,39.492646,2.062126,35.8834,39.8468,39.8601,41.2034,41.2034,5.3200
HOME_&_GARDEN_1_259,19040.0,12.140643,2.010872,7.4875,9.3500,13.7125,13.7125,14.4000,6.9125
ACCESORIES_1_311,17843.0,20.972076,1.994711,17.2900,19.4978,19.4978,23.6474,23.9134,6.6234
HOME_&_GARDEN_2_446,18984.0,31.816621,1.854610,18.7375,31.1500,32.4625,32.4625,35.0000,16.2625


1. Rango de precios (max - min): mide la diferencia entre la tienda más cara y la más barata para cada producto. Un rango alto indica que el mismo producto se vende a precios muy diferentes según la ubicación.
2. Desviación estándar: mide cuánto varían los precios en general. Mientras que el rango solo captura los extremos, la desviación estándar considera todos los precios y detecta si la variación es generalizada o si es solo una tienda atípica.

El rango solo mira extremos y la desviación penaliza más cuando MUCHOS precios están dispersos. Yo le preguntaría a Dani porque no sé cual nos puede interesar más

### productos estacionales

In [52]:
ventas_estacion = df.groupby(['item', 'season'])['sales'].sum().reset_index()
ventas_estacion['total_producto'] = ventas_estacion.groupby('item')['sales'].transform('sum')
ventas_estacion['pct'] = (ventas_estacion['sales'] / ventas_estacion['total_producto'] * 100).round(1)
# ventas_estacion
resumen_estaciones = ventas_estacion.pivot(index='item', columns='season', values='pct')
resumen_estaciones['max_season'] = resumen_estaciones.max(axis=1)
productos_estacionales = resumen_estaciones.sort_values('max_season', ascending=False).head(20)
print("Productos más estacionales (concentrados en una época):")
print(productos_estacionales)

Productos más estacionales (concentrados en una época):
season               Invierno  Otoño  Primavera  Verano  max_season
item                                                               
HOME_&_GARDEN_1_297       4.5   13.3       20.1    62.1        62.1
HOME_&_GARDEN_2_162       0.3   23.9       16.0    59.8        59.8
HOME_&_GARDEN_2_340       2.8    5.1       32.6    59.6        59.6
HOME_&_GARDEN_1_189       4.0   18.4       19.6    58.0        58.0
HOME_&_GARDEN_2_022      11.0    6.4       57.9    24.6        57.9
HOME_&_GARDEN_1_049       0.1    3.8       38.5    57.6        57.6
HOME_&_GARDEN_2_449       4.3    9.7       28.4    57.6        57.6
HOME_&_GARDEN_2_289       2.7   15.9       24.0    57.4        57.4
HOME_&_GARDEN_2_344       6.1   17.8       19.7    56.4        56.4
HOME_&_GARDEN_2_080       6.3    7.8       29.8    56.1        56.1
HOME_&_GARDEN_2_477       4.6   19.7       21.1    54.6        54.6
SUPERMARKET_3_350        14.7   13.6       54.1    17.5     

In [53]:
top_estacionales = productos_estacionales.head(5)
fig = make_subplots(rows=3, cols=2, subplot_titles=[f"{prod}" for prod in top_estacionales.index[:5]])

for idx, prod in enumerate(top_estacionales.index[:5]):
    datos_prod = df[df['item'] == prod].groupby('date')['sales'].sum().reset_index()
    fig.add_trace(go.Scatter(x=datos_prod['date'], y=datos_prod['sales'], mode='lines', name=prod, showlegend=False), row=idx//2+1, col=idx%2+1)

fig.update_layout(height=900, title_text="Evolución Temporal - Top 5 Productos Estacionales")
fig.show()